# Titanic Survival Analysis — Task 2
### Data Science / Data Analysis with Python Internship — Main Crafts Technology

**Workflow:** Load → Clean → Analyze → Visualize → Conclude

Dataset: `train.csv` — the classic Titanic passenger dataset (891 passengers, Kaggle).


## 1. Load Dataset

We load the Titanic dataset from **Kaggle**: downloaded manually from the [Kaggle Titanic competition page](https://www.kaggle.com/c/titanic/data) (`train.csv`), then uploaded into this Colab session.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Upload train.csv downloaded from Kaggle (https://www.kaggle.com/c/titanic/data)
from google.colab import files
uploaded = files.upload()

# Load into pandas
df = pd.read_csv('train.csv')

# Preview
df.head()

Saving train.csv to train.csv


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [2]:
print("Shape of dataset:", df.shape)
df.info()

Shape of dataset: (891, 12)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    object 
 4   Sex          891 non-null    object 
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    object 
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    object 
 11  Embarked     889 non-null    object 
dtypes: float64(2), int64(5), object(5)
memory usage: 83.7+ KB


## 2. Explore & Clean Data

Unlike Task 1's dataset, the Titanic dataset has real missing values. We check for them, then handle the most important one: **Age**.

In [3]:
# Check for missing values
print("Missing values per column:")
print(df.isnull().sum())

Missing values per column:
PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age            177
SibSp            0
Parch            0
Ticket           0
Fare             0
Cabin          687
Embarked         2
dtype: int64


**Observation:** `Age` has 177 missing values, `Cabin` has 687 (mostly empty), and `Embarked` has 2. Since `Age` is important for our analysis (survival rate by age group), we fill missing ages with the **median age** rather than dropping those rows, which would lose valuable data. `Cabin` has too many missing values to be useful here, so we drop it. `Embarked` has very few missing rows, so we fill them with the most common port.

In [4]:
# Handle missing Age values -> fill with median age
df['Age'] = df['Age'].fillna(df['Age'].median())

# Drop Cabin column (too many missing values to be useful)
df = df.drop(columns=['Cabin'])

# Fill missing Embarked values with the most common port (mode)
df['Embarked'] = df['Embarked'].fillna(df['Embarked'].mode()[0])

# Check for duplicate rows
print("Duplicate rows:", df.duplicated().sum())

# Confirm no missing values remain (except intentionally dropped Cabin)
print("\nMissing values after cleaning:")
print(df.isnull().sum())

Duplicate rows: 0

Missing values after cleaning:
PassengerId    0
Survived       0
Pclass         0
Name           0
Sex            0
Age            0
SibSp          0
Parch          0
Ticket         0
Fare           0
Embarked       0
dtype: int64


In [5]:
df.describe()

,PassengerId,Survived,Pclass,Age,SibSp,Parch,Fare
count,891.000000,891.000000,891.000000,891.000000,891.000000,891.000000,891.000000
mean,446.000000,0.383838,2.308642,29.361582,0.523008,0.381594,32.204208
std,257.353842,0.486592,0.836071,13.019697,1.102743,0.806057,49.693429
min,1.000000,0.000000,1.000000,0.420000,0.000000,0.000000,0.000000
25%,223.500000,0.000000,2.000000,22.000000,0.000000,0.000000,7.910400
50%,446.000000,0.000000,3.000000,28.000000,0.000000,0.000000,14.454200
75%,668.500000,1.000000,3.000000,35.000000,1.000000,0.000000,31.000000
max,891.000000,1.000000,3.000000,80.000000,8.000000,6.000000,512.329200


**Observation:** After cleaning, the dataset has no missing values in the columns we need. We have 891 passenger records with details like age, sex, passenger class, fare, and survival outcome (`Survived`: 1 = survived, 0 = did not survive).